# Classical Hardness Audit for the Blob Defect Dataset

This notebook asks a blunt question before spending quantum runtime: **is the current synthetic defect dataset actually classically hard?**

The answer is likely no for the current generator. A defective image contains one near-black blob while clean images stay mid-gray. That means a one-line rule such as "does this image have a very dark pixel?" may solve the task. If that happens, a quantum classifier can still be a fun demo, but it is not a good quantum-advantage story.

Use this notebook as the first gate. If simple rules win, move to the quantum-tailored notebooks rather than optimizing a QNN on an easy target.

## 1. Imports and Local Paths

The imports mirror `defect_qml_experiments.ipynb` so that the notebooks feel consistent and can use the same local defect generator.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

# Resolve the util folder robustly from either the repo root or this notebook folder.
UTIL_DIR = Path("notebooks/iqucodefest-2026-main/main_challenge/util")
if not UTIL_DIR.exists():
    UTIL_DIR = Path.cwd() / "util"
UTIL_DIR = UTIL_DIR.resolve()

if str(UTIL_DIR) not in sys.path:
    sys.path.insert(0, str(UTIL_DIR))

from defect_gen import generate_dataset, show_samples

SEED = 8398
np.random.seed(SEED)

print(f"Using defect generator from: {UTIL_DIR}")

## 2. Generate the Current Blob Dataset

These settings intentionally match the current notebook: mid-gray normal pixels and a near-black defect blob. The high contrast is useful for debugging, but it can make the classification task trivial.

In [ ]:
N_SAMPLES = 400
IMAGE_SIZE = 8
DEFECT_RATE = 0.10

images, labels = generate_dataset(
    n_samples=N_SAMPLES,
    size=IMAGE_SIZE,
    defect_rate=DEFECT_RATE,
    normal_range=(0.3, 0.5),
    defect_range=(0.95, 0.99),
    seed=SEED,
)

X = images.reshape(len(images), -1)
y = labels

print("images shape:", images.shape)
print("class balance:", {"clean": int((y == 0).sum()), "defect": int((y == 1).sum())})

## 3. Visual Check

The defects are visually obvious here. That is a useful clue: if humans can spot the label from a single dark patch, classical baselines usually can too.

In [ ]:
show_samples(images, labels, n_show=12, cols=6)

## 4. Train/Test Split

Keep the split stratified because defects are rare. The audit is not about ordinary accuracy; it is about defect recall, F1, and balanced accuracy.

In [ ]:
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X,
    y,
    images,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

print("train:", X_train.shape, "clean:", int((y_train == 0).sum()), "defect:", int((y_train == 1).sum()))
print("test :", X_test.shape, "clean:", int((y_test == 0).sum()), "defect:", int((y_test == 1).sum()))

## 5. Shared Evaluation Helpers

Balanced accuracy gives equal weight to clean and defect classes. Defect F1 punishes missed defects and false alarms.

In [ ]:
def evaluate_predictions(name, y_true, y_pred, *, show_confusion=True):
    """Print metrics that are useful for rare-defect classification."""
    print(f"\n{name}")
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect F1 score  :", f1_score(y_true, y_pred, zero_division=0))
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=True):
    y_pred = model.predict(X_eval)
    evaluate_predictions(name, y_eval, y_pred, show_confusion=show_confusion)
    return y_pred

## 6. A One-Feature Threshold Rule

Because the generator uses the convention `0 = white`, `1 = black`, a defect creates a pixel close to `1.0`. This threshold model only looks at the darkest pixel in the image.

In [ ]:
def image_audit_features(flat_X):
    """Tiny hand-built features that should not solve a genuinely hard problem."""
    max_darkness = flat_X.max(axis=1)
    mean_darkness = flat_X.mean(axis=1)
    std_darkness = flat_X.std(axis=1)
    top4_mean = np.sort(flat_X, axis=1)[:, -4:].mean(axis=1)
    return np.column_stack([max_darkness, mean_darkness, std_darkness, top4_mean])

simple_train = image_audit_features(X_train)
simple_test = image_audit_features(X_test)

# Choose the threshold that maximizes balanced accuracy on the training split.
threshold_grid = np.linspace(simple_train[:, 0].min(), simple_train[:, 0].max(), 300)
threshold_scores = []
for threshold in threshold_grid:
    pred = (simple_train[:, 0] >= threshold).astype(int)
    threshold_scores.append(balanced_accuracy_score(y_train, pred))

best_threshold = float(threshold_grid[int(np.argmax(threshold_scores))])
print("best max-pixel threshold:", best_threshold)

threshold_pred = (simple_test[:, 0] >= best_threshold).astype(int)
evaluate_predictions("Single max-pixel threshold", y_test, threshold_pred)

## 7. Small Classical Models on Simple Features

If these models win with four hand-built features, the quantum method is solving an easy proxy rather than showing an advantage.

In [ ]:
simple_models = {
    "Logistic Regression on 4 audit features": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]),
    "RBF SVM on 4 audit features": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Random Forest on 4 audit features": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in simple_models.items():
    model.fit(simple_train, y_train)
    evaluate_classifier(name, model, simple_test, y_test, show_confusion=False)

## 8. Classical Models on Raw Pixels

These are the same sort of sanity baselines used in `defect_qml_experiments.ipynb`. The four-feature audit above is more important for hardness, but raw-pixel models help confirm the picture.

In [ ]:
raw_models = {
    "Raw-pixel Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ]),
    "Raw-pixel RBF SVM": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
    ]),
    "Raw-pixel Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in raw_models.items():
    model.fit(X_train, y_train)
    evaluate_classifier(name, model, X_test, y_test, show_confusion=False)

## 9. Make the Generator Less Obvious

This stress test moves the defect intensity closer to the normal range. It is still a local-blob task, so it is not truly classically hard, but it helps tune difficulty for demos.

In [ ]:
stress_images, stress_labels = generate_dataset(
    n_samples=500,
    size=IMAGE_SIZE,
    defect_rate=0.10,
    normal_range=(0.30, 0.50),
    defect_range=(0.48, 0.56),
    shapes=[(1, 1)],
    seed=SEED + 1,
)

stress_X = stress_images.reshape(len(stress_images), -1)
stress_y = stress_labels

Xtr, Xte, ytr, yte = train_test_split(
    stress_X,
    stress_y,
    test_size=0.25,
    random_state=SEED,
    stratify=stress_y,
)

stress_simple_tr = image_audit_features(Xtr)
stress_simple_te = image_audit_features(Xte)

stress_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
])
stress_model.fit(stress_simple_tr, ytr)
evaluate_classifier("Stress test: RBF SVM on 4 audit features", stress_model, stress_simple_te, yte)

## 10. Decision

If the threshold or four-feature models perform well, do not sell this dataset as classically hard. The better hackathon direction is:

- Use the blob dataset as a clear baseline and visualization tool.
- Build a second, quantum-tailored dataset where the label depends on global correlations, not one dark pixel.
- Lead with quantum kernels or projected quantum kernels because they give a cleaner quantum-advantage narrative than a small variational QNN.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`
- `10_heat_exchanger_network_qubo_qaoa.ipynb`
- `11_Final_QCNN_rare_defect_detection.ipynb`

References used for the quantum-ML and optimization direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Perez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlicek et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Scholkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Furman and Sahinidis, "Computational complexity of heat exchanger network synthesis," Computers & Chemical Engineering 25(9-10), 1371-1390 (2001): https://doi.org/10.1016/S0098-1354(01)00681-0
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html